In [ ]:
!pip install transformers datasets==3.6.0 huggingface_hub tqdm

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from datasets import load_dataset
from huggingface_hub import login, create_repo, HfApi, hf_hub_download
import numpy as np
import json
from tqdm import tqdm
from collections import defaultdict

### Global Configuration

In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "HF_TOKEN"
hf_token = UserSecretsClient().get_secret(secret_label)

class Config:
    # Experiment Setup - Set which experiments to run
    TRAIN_TOKEN_CHOICE = True          # Train Token Choice router
    TRAIN_HASH = False              # Train Hash router
    TRAIN_WITH_LOAD_BALANCER = True    # Train with load balancer
    TRAIN_WITHOUT_LOAD_BALANCER = False # Train without load balancer
    USE_CUSTOM_GQA = False              # Use custom GQA (Bonus 2)
    USE_LORA = True                     # Use LoRA for experts (Bonus 3)
    
    # Model Architecture
    D_MODEL = 512
    NHEAD = 8
    NUM_ENCODER_LAYERS = 6
    NUM_DECODER_LAYERS = 6
    NUM_EXPERTS = 4
    D_FF = 1024
    TOP_K = 2
    DROPOUT = 0.1
    MAX_SEQ_LEN = 512
    GQA_NUM_KV_HEADS = 4  # For GQA (must divide NHEAD)
    
    # LoRA Configuration (Bonus 3)
    LORA_RANK = 16                      # LoRA rank (lower = more compression)
    LORA_ALPHA = 32                     # LoRA scaling factor
    LORA_DROPOUT = 0.1                  # LoRA dropout
    
    #Checkpoint Continuation
    CONTINUE_FROM_CHECKPOINT = True  # Set to True to load from HF
    CHECKPOINT_REPO_MAPPING = {
        'token_choice': 'J10Official/sparse-moe-Token-Choice-NoLB',
        'hash': 'J10Official/sparse-moe-Hash-NoLB',
        'token_choice_with_LB': 'J10Official/sparse-moe-Token-Choice',
        'hash_with_LB': 'J10Official/sparse-moe-Hash',
        # GQA versions
        'token_choice_gqa': 'J10Official/sparse-moe-Token-Choice-NoLB-GQA',
        'hash_gqa': 'J10Official/sparse-moe-Hash-NoLB-GQA',
        'token_choice_with_LB_gqa': 'J10Official/sparse-moe-Token-Choice-GQA',
        'hash_with_LB_gqa': 'J10Official/sparse-moe-Hash-GQA',
        # LoRA versions
        'token_choice_lora': 'J10Official/sparse-moe-Token-Choice-NoLB-LoRA',
        'hash_lora': 'J10Official/sparse-moe-Hash-NoLB-LoRA',
        'token_choice_with_LB_lora': 'J10Official/sparse-moe-Token-Choice-LoRA',
        'hash_with_LB_lora': 'J10Official/sparse-moe-Hash-LoRA',
        # GQA + LoRA combinations
        'token_choice_gqa_lora': 'J10Official/sparse-moe-Token-Choice-NoLB-GQA-LoRA',
        'hash_gqa_lora': 'J10Official/sparse-moe-Hash-NoLB-GQA-LoRA',
        'token_choice_with_LB_gqa_lora': 'J10Official/sparse-moe-Token-Choice-GQA-LoRA',
        'hash_with_LB_gqa_lora': 'J10Official/sparse-moe-Hash-GQA-LoRA',
    }
    
    # Load Balancing
    LOAD_BALANCE_ALPHA = 0.01
    
    # Training Settings
    BATCH_SIZE = 32
    NUM_EPOCHS = 1
    LEARNING_RATE = 5e-5
    WARMUP_STEPS = 500
    GRAD_CLIP = 1.0
    MAX_LENGTH = 256
    
    # Dataset
    DATASET_NAME = "EdinburghNLP/xsum"
    TRAIN_SAMPLES = 100000  # Set to None for full dataset
    VAL_SAMPLES = 2000
    TEST_SAMPLES = 2000
    
    # Tokenizer
    TOKENIZER_NAME = "t5-small"
    
    # Checkpointing
    CHECKPOINT_DIR = "./checkpoints"
    SAVE_EVERY_N_EPOCHS = 1
    
    # Hugging Face
    HF_USERNAME = "J10Official" 
    PUSH_TO_HUB = True 
    
    # Evaluation
    EVAL_SAMPLES = 500
    GENERATION_MAX_LENGTH = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Display LoRA configuration if enabled
if Config.USE_LORA:
    print(f"\n📡 LoRA Configuration Enabled:")
    print(f"   Rank: {Config.LORA_RANK}")
    print(f"   Alpha: {Config.LORA_ALPHA}")
    print(f"   Dropout: {Config.LORA_DROPOUT}")
    print(f"   Expected parameter reduction: ~{(1 - Config.LORA_RANK * 2 / (Config.D_FF + Config.D_MODEL)) * 100:.1f}%")

### Grouped Query Attentions

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Custom GQA implementation from scratch."""
    def __init__(self, d_model, num_heads, num_kv_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_heads
        self.num_queries_per_kv = num_heads // num_kv_heads
        
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5
        
    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
        B, T, _ = query.shape
        
        # Project
        Q = self.q_proj(query).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(key).view(B, -1, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(value).view(B, -1, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # Repeat K,V for query groups
        if self.num_queries_per_kv > 1:
            K = K.repeat_interleave(self.num_queries_per_kv, dim=1)
            V = V.repeat_interleave(self.num_queries_per_kv, dim=1)
        
        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        if key_padding_mask is not None:
            scores = scores.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.out_proj(out)
        
        return out, attn.mean(dim=1)

### LoRA Implementation (Low-Rank Adaptation) - Bonus 3

In [ ]:
class LoRALinear(nn.Module):
    """
    LoRA (Low-Rank Adaptation) Linear layer implementation from scratch.
    
    Implementation Challenges & Solutions:
    1. Rank selection: Too low rank may hurt performance, too high reduces efficiency
       Solution: Make rank configurable, start with 16 (good balance)
    2. Scaling factor (alpha): Needs careful tuning for stability
       Solution: Use alpha/rank scaling as in original LoRA paper
    3. Initialization: LoRA weights need proper initialization
       Solution: Zero-init B, normal-init A for stable training start
    4. Training dynamics: LoRA can be sensitive to learning rates
       Solution: Freeze base weights, only train LoRA parameters
    5. Memory overhead: While LoRA reduces parameters, it adds computational overhead
       Solution: Proper memory management and GPU optimization
    """
    def __init__(self, in_features, out_features, rank=16, alpha=32, dropout=0.1):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # Original frozen weights (these will be frozen during training)
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        
        # LoRA decomposition: W = W_frozen + (B @ A) * scaling
        # A: down-projection (in_features -> rank)  
        # B: up-projection (rank -> out_features)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))  # Zero init for B
        
        self.dropout = nn.Dropout(dropout)
        
        # Initialize LoRA weights
        self._init_weights()
        
    def _init_weights(self):
        """Initialize LoRA weights following best practices."""
        # Initialize original weight with Xavier/Glorot initialization
        nn.init.xavier_uniform_(self.weight)
        
        # Initialize A with small random values (Gaussian)
        nn.init.normal_(self.lora_A, mean=0.0, std=0.02)
        
        # Initialize B with zeros (standard practice)
        nn.init.zeros_(self.lora_B)
    
    def forward(self, x):
        """Forward pass with LoRA adaptation."""
        # Original linear transformation (frozen)
        base_output = F.linear(x, self.weight, self.bias)
        
        # LoRA adaptation path: x -> A -> dropout -> B -> scale
        lora_output = F.linear(x, self.lora_A.T)  # Down-project
        lora_output = self.dropout(lora_output)
        lora_output = F.linear(lora_output, self.lora_B.T)  # Up-project
        lora_output = lora_output * self.scaling
        
        return base_output + lora_output
    
    def freeze_base_weights(self):
        """Freeze the original weights, only train LoRA parameters."""
        self.weight.requires_grad_(False)
        self.bias.requires_grad_(False)
    
    def unfreeze_all(self):
        """Unfreeze all weights for full fine-tuning."""
        self.weight.requires_grad_(True)
        self.bias.requires_grad_(True)
    
    def get_parameter_count(self):
        """Get parameter counts for efficiency analysis."""
        base_params = self.weight.numel() + self.bias.numel()
        lora_params = self.lora_A.numel() + self.lora_B.numel()
        return {
            'base_parameters': base_params,
            'lora_parameters': lora_params,
            'total_parameters': base_params + lora_params,
            'trainable_parameters': lora_params if not self.weight.requires_grad else base_params + lora_params,
            'parameter_efficiency': lora_params / (base_params + lora_params)
        }

class LoRAExpert(nn.Module):
    """Expert network using LoRA for parameter efficiency."""
    def __init__(self, d_model, d_ff, use_lora=True, lora_rank=16, lora_alpha=32, lora_dropout=0.1):
        super().__init__()
        self.use_lora = use_lora
        
        if use_lora:
            # LoRA-based expert
            self.linear1 = LoRALinear(d_model, d_ff, lora_rank, lora_alpha, lora_dropout)
            self.linear2 = LoRALinear(d_ff, d_model, lora_rank, lora_alpha, lora_dropout)
            # Freeze base weights, only train LoRA parameters
            self.linear1.freeze_base_weights()
            self.linear2.freeze_base_weights()
        else:
            # Standard expert (for comparison)
            self.linear1 = nn.Linear(d_model, d_ff)
            self.linear2 = nn.Linear(d_ff, d_model)
        
        self.activation = nn.ReLU()
        
    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        return x
    
    def get_parameter_count(self):
        """Return parameter count information."""
        if self.use_lora:
            # Get LoRA parameter information
            params1 = self.linear1.get_parameter_count()
            params2 = self.linear2.get_parameter_count()
            return {
                'total': params1['total_parameters'] + params2['total_parameters'],
                'trainable': params1['trainable_parameters'] + params2['trainable_parameters'],
                'efficiency': (params1['trainable_parameters'] + params2['trainable_parameters']) / 
                             (params1['total_parameters'] + params2['total_parameters'])
            }
        else:
            # Standard parameter count
            total_params = sum(p.numel() for p in self.parameters())
            return {
                'total': total_params,
                'trainable': total_params,
                'efficiency': 1.0
            }

def compare_parameter_efficiency():
    """Utility function to compare parameter counts between standard and LoRA experts."""
    d_model, d_ff = Config.D_MODEL, Config.D_FF
    
    # Standard expert
    standard_expert = LoRAExpert(d_model, d_ff, use_lora=False)
    standard_counts = standard_expert.get_parameter_count()
    
    # LoRA expert
    lora_expert = LoRAExpert(d_model, d_ff, use_lora=True, 
                            lora_rank=Config.LORA_RANK, lora_alpha=Config.LORA_ALPHA)
    lora_counts = lora_expert.get_parameter_count()
    
    print("📊 Parameter Efficiency Comparison:")
    print(f"  Standard Expert: {standard_counts['total']:,} total, {standard_counts['trainable']:,} trainable")
    print(f"  LoRA Expert: {lora_counts['total']:,} total, {lora_counts['trainable']:,} trainable")
    print(f"  Parameter Reduction: {(1 - lora_counts['trainable']/standard_counts['trainable'])*100:.1f}%")
    print(f"  LoRA Efficiency: {lora_counts['efficiency']*100:.1f}% trainable")
    
    return standard_counts, lora_counts

# Test LoRA implementation
print("🔬 LoRA Implementation Ready!")
if Config.USE_LORA:
    print(f"LoRA Configuration: rank={Config.LORA_RANK}, alpha={Config.LORA_ALPHA}, dropout={Config.LORA_DROPOUT}")
    compare_parameter_efficiency()
else:
    print("LoRA disabled in config. Set Config.USE_LORA = True to enable.")

### Load from Checkpoints

In [ ]:
def load_checkpoint_from_hf(model, repo_id, checkpoint_name="Token-Choice-NoLB_best.pt"):
    """
    Load model checkpoint from Hugging Face Hub
    """
    try:
        print(f"\n Attempting to load checkpoint from {repo_id}...")
        
        # Download the checkpoint file
        checkpoint_path = hf_hub_download(
            repo_id=repo_id,
            filename=checkpoint_name,
            cache_dir=Config.CHECKPOINT_DIR
        )
        
        # Load the checkpoint
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        
        # Load model state
        model.load_state_dict(checkpoint['model_state_dict'])
        
        print(f"Successfully loaded checkpoint from epoch {checkpoint['epoch']}")
        print(f"Previous validation loss: {checkpoint['val_loss']:.4f}")
        
        return model, checkpoint['epoch'], checkpoint['val_loss']
        
    except Exception as e:
        print(f"Failed to load checkpoint from HF: {e}")
        print(f"Starting training from scratch...")
        return model, 0, float('inf')

### Sparse MoE Components

In [ ]:
class HashRouter(nn.Module):
    """Hash-based routing."""
    def __init__(self, num_experts, d_model):
        super().__init__()
        self.num_experts = num_experts
        self.hash_matrix = nn.Parameter(torch.randn(d_model, num_experts), requires_grad=False)
        
    def forward(self, x):
        scores = torch.matmul(x, self.hash_matrix)
        indices = torch.argmax(scores, dim=-1)
        weights = F.one_hot(indices, self.num_experts).float()
        return weights, indices

class TokenChoiceRouter(nn.Module):
    """Token-choice Top-K routing."""
    def __init__(self, num_experts, d_model, top_k):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, num_experts)
        
    def forward(self, x):
        logits = self.gate(x)
        top_k_logits, top_k_indices = torch.topk(logits, self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_logits, dim=-1)
        
        weights = torch.zeros_like(logits)
        weights.scatter_(-1, top_k_indices, top_k_weights)
        return weights, top_k_indices

class SparseMoELayer(nn.Module):
    """Sparse MoE layer - TRUE SPARSE dispatch with optional LoRA experts."""
    def __init__(self, d_model, num_experts, d_ff, router_type, top_k=2, use_lora=False):
        super().__init__()
        self.num_experts = num_experts
        self.d_model = d_model
        self.use_lora = use_lora
        
        # Experts - can be LoRA or standard
        if use_lora:
            print(f"  Creating LoRA experts (rank={Config.LORA_RANK}, alpha={Config.LORA_ALPHA})")
            self.experts = nn.ModuleList([
                LoRAExpert(d_model, d_ff, use_lora=True, 
                          lora_rank=Config.LORA_RANK,
                          lora_alpha=Config.LORA_ALPHA, 
                          lora_dropout=Config.LORA_DROPOUT)
                for _ in range(num_experts)
            ])
        else:
            self.experts = nn.ModuleList([
                nn.Sequential(
                    nn.Linear(d_model, d_ff),
                    nn.ReLU(),
                    nn.Linear(d_ff, d_model)
                ) for _ in range(num_experts)
            ])
        
        # Router
        if router_type == 'hash':
            self.router = HashRouter(num_experts, d_model)
        else:
            self.router = TokenChoiceRouter(num_experts, d_model, top_k)
    
    def forward(self, x, use_load_balancer=True, alpha=0.01):
        B, T, D = x.shape
        
        # Route
        weights, indices = self.router(x)
        
        # Load balancing loss
        if use_load_balancer:
            expert_usage = weights.sum(dim=(0, 1)) / (B * T)
            target = 1.0 / self.num_experts
            load_loss = alpha * torch.sum((expert_usage - target) ** 2)
        else:
            load_loss = torch.tensor(0.0, device=x.device)
        
        # SPARSE dispatch
        x_flat = x.view(-1, D)
        weights_flat = weights.view(-1, self.num_experts)
        out_flat = torch.zeros_like(x_flat)
        
        for expert_idx in range(self.num_experts):
            mask = weights_flat[:, expert_idx] > 0
            if mask.any():
                tokens = x_flat[mask]
                expert_out = self.experts[expert_idx](tokens)
                expert_weights = weights_flat[mask, expert_idx:expert_idx+1]
                out_flat[mask] += expert_out * expert_weights
        
        output = out_flat.view(B, T, D)
        return output, load_loss, weights
    
    def get_expert_parameters(self):
        """Get parameter counts for each expert (useful for LoRA analysis)."""
        if hasattr(self.experts[0], 'get_parameter_count'):
            return [expert.get_parameter_count() for expert in self.experts]
        else:
            return [sum(p.numel() for p in expert.parameters() if p.requires_grad) 
                   for expert in self.experts]
    
    def get_parameter_summary(self):
        """Get overall parameter summary for the MoE layer."""
        expert_params = self.get_expert_parameters()
        if isinstance(expert_params[0], dict):  # LoRA experts
            total_params = sum(ep['total'] for ep in expert_params)
            trainable_params = sum(ep['trainable'] for ep in expert_params)
            return {
                'total_parameters': total_params,
                'trainable_parameters': trainable_params,
                'frozen_parameters': total_params - trainable_params,
                'efficiency': trainable_params / total_params if total_params > 0 else 0,
                'use_lora': self.use_lora
            }
        else:  # Standard experts
            total_params = sum(expert_params)
            return {
                'total_parameters': total_params,
                'trainable_parameters': total_params,
                'frozen_parameters': 0,
                'efficiency': 1.0,
                'use_lora': self.use_lora
            }

### MoE Components

In [ ]:
class MoEEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, num_experts, d_ff, router_type, top_k, dropout, use_gqa, use_lora=False):
        super().__init__()
        if use_gqa:
            self.attn = GroupedQueryAttention(d_model, nhead, Config.GQA_NUM_KV_HEADS, dropout)
        else:
            self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        
        self.moe = SparseMoELayer(d_model, num_experts, d_ff, router_type, top_k, use_lora)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None, use_load_balancer=True):
        attn_out, _ = self.attn(x, x, x, attn_mask=mask, key_padding_mask=None)
        x = self.norm1(x + self.dropout(attn_out))
        
        moe_out, load_loss, weights = self.moe(x, use_load_balancer, Config.LOAD_BALANCE_ALPHA)
        x = self.norm2(x + self.dropout(moe_out))
        
        return x, load_loss, weights

class MoEDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, num_experts, d_ff, router_type, top_k, dropout, use_gqa, use_lora=False):
        super().__init__()
        if use_gqa:
            self.self_attn = GroupedQueryAttention(d_model, nhead, Config.GQA_NUM_KV_HEADS, dropout)
            self.cross_attn = GroupedQueryAttention(d_model, nhead, Config.GQA_NUM_KV_HEADS, dropout)
        else:
            self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        
        self.moe = SparseMoELayer(d_model, num_experts, d_ff, router_type, top_k, use_lora)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, memory, tgt_mask=None, memory_mask=None, use_load_balancer=True):
        self_attn_out, _ = self.self_attn(x, x, x, attn_mask=tgt_mask, key_padding_mask=None)
        x = self.norm1(x + self.dropout(self_attn_out))
        
        cross_attn_out, _ = self.cross_attn(x, memory, memory, attn_mask=memory_mask, key_padding_mask=None)
        x = self.norm2(x + self.dropout(cross_attn_out))
        
        moe_out, load_loss, weights = self.moe(x, use_load_balancer, Config.LOAD_BALANCE_ALPHA)
        x = self.norm3(x + self.dropout(moe_out))
        
        return x, load_loss, weights

class MoETransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_enc, num_dec, num_experts, 
                 d_ff, router_type, top_k, dropout, max_len, use_gqa, use_lora=False):
        super().__init__()
        self.d_model = d_model
        self.use_lora = use_lora
        
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        
        self.encoder_layers = nn.ModuleList([
            MoEEncoderLayer(d_model, nhead, num_experts, d_ff, router_type, top_k, dropout, use_gqa, use_lora)
            for _ in range(num_enc)
        ])
        
        self.decoder_layers = nn.ModuleList([
            MoEDecoderLayer(d_model, nhead, num_experts, d_ff, router_type, top_k, dropout, use_gqa, use_lora)
            for _ in range(num_dec)
        ])
        
        self.output_proj = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src, tgt, use_load_balancer=True):
        # Encode
        src_emb = self.embed(src) * np.sqrt(self.d_model)
        src_pos = self.pos_embed(torch.arange(src.size(1), device=src.device))
        src_emb = self.dropout(src_emb + src_pos)
        
        memory = src_emb
        total_load_loss = 0
        
        for layer in self.encoder_layers:
            memory, load_loss, _ = layer(memory, use_load_balancer=use_load_balancer)
            total_load_loss += load_loss
        
        # Decode
        tgt_emb = self.embed(tgt) * np.sqrt(self.d_model)
        tgt_pos = self.pos_embed(torch.arange(tgt.size(1), device=tgt.device))
        tgt_emb = self.dropout(tgt_emb + tgt_pos)
        
        output = tgt_emb
        for layer in self.decoder_layers:
            output, load_loss, _ = layer(output, memory, use_load_balancer=use_load_balancer)
            total_load_loss += load_loss
        
        logits = self.output_proj(output)
        return logits, total_load_loss
    
    def get_parameter_summary(self):
        """Get comprehensive parameter summary including LoRA efficiency."""
        encoder_summaries = [layer.moe.get_parameter_summary() for layer in self.encoder_layers]
        decoder_summaries = [layer.moe.get_parameter_summary() for layer in self.decoder_layers]
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        moe_total = sum(s['total_parameters'] for s in encoder_summaries + decoder_summaries)
        moe_trainable = sum(s['trainable_parameters'] for s in encoder_summaries + decoder_summaries)
        moe_frozen = sum(s['frozen_parameters'] for s in encoder_summaries + decoder_summaries)
        
        return {
            'total_parameters': total_params,
            'trainable_parameters': trainable_params,
            'frozen_parameters': total_params - trainable_params,
            'moe_total_parameters': moe_total,
            'moe_trainable_parameters': moe_trainable,
            'moe_frozen_parameters': moe_frozen,
            'overall_efficiency': trainable_params / total_params if total_params > 0 else 0,
            'moe_efficiency': moe_trainable / moe_total if moe_total > 0 else 0,
            'use_lora': self.use_lora
        }

### Dataset

In [ ]:
def load_data(tokenizer):
    print(" Loading dataset...")
    dataset = load_dataset(Config.DATASET_NAME)
    
    # Subset
    train_size = Config.TRAIN_SAMPLES or len(dataset['train'])
    val_size = Config.VAL_SAMPLES or len(dataset['validation'])
    test_size = Config.TEST_SAMPLES or len(dataset['test'])
    
    train_data = dataset['train'].select(range(min(train_size, len(dataset['train']))))
    val_data = dataset['validation'].select(range(min(val_size, len(dataset['validation']))))
    test_data = dataset['test'].select(range(min(test_size, len(dataset['test']))))
    
    def collate(batch):
        docs = [item['document'] for item in batch]
        sums = [item['summary'] for item in batch]
        
        src = tokenizer(docs, padding=True, truncation=True, max_length=Config.MAX_LENGTH, return_tensors='pt')
        tgt = tokenizer(sums, padding=True, truncation=True, max_length=Config.MAX_LENGTH, return_tensors='pt')
        
        return {
            'src_ids': src['input_ids'],
            'src_mask': src['attention_mask'],
            'tgt_ids': tgt['input_ids'],
            'tgt_mask': tgt['attention_mask']
        }
    
    train_loader = DataLoader(train_data, batch_size=Config.BATCH_SIZE, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_data, batch_size=Config.BATCH_SIZE, collate_fn=collate)
    test_loader = DataLoader(test_data, batch_size=Config.BATCH_SIZE, collate_fn=collate)
    
    return train_loader, val_loader, test_loader, test_data


### Training

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device, vocab_size, pad_token_id, use_load_balancer=True):
    model.train()
    total_loss = 0
    total_load_loss = 0
    total_nll_loss = 0
    total_tokens = 0
    correct_predictions = 0
    
    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Training")):
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        # Prepare decoder input and target
        decoder_input = labels[:, :-1]
        target = labels[:, 1:]
        
        # Forward pass
        logits, load_loss = model(input_ids, decoder_input, use_load_balancer)
        
        # Calculate NLL loss
        nll_loss = criterion(logits.reshape(-1, vocab_size), target.reshape(-1))
        
        # Total loss includes load balancing
        total_model_loss = nll_loss + load_loss
        
        total_model_loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Accumulate losses
        total_loss += total_model_loss.item()
        total_load_loss += load_loss.item()
        total_nll_loss += nll_loss.item()
        
        # Calculate accuracy
        mask = (target != pad_token_id)
        total_tokens += mask.sum().item()
        predictions = logits.argmax(dim=-1)
        correct_predictions += ((predictions == target) & mask).sum().item()
        
    avg_loss = total_loss / len(dataloader)
    avg_load_loss = total_load_loss / len(dataloader)
    avg_nll_loss = total_nll_loss / len(dataloader)
    accuracy = correct_predictions / total_tokens if total_tokens > 0 else 0
    
    return avg_loss, avg_load_loss, avg_nll_loss, accuracy

def evaluate_model(model, dataloader, criterion, device, vocab_size, pad_token_id, use_load_balancer=True):
    model.eval()
    total_loss = 0
    total_load_loss = 0
    total_nll_loss = 0
    total_tokens = 0
    correct_predictions = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            # Prepare decoder input and target
            decoder_input = labels[:, :-1]
            target = labels[:, 1:]
            
            # Forward pass
            logits, load_loss = model(input_ids, decoder_input, use_load_balancer)
            
            # Calculate NLL loss
            nll_loss = criterion(logits.reshape(-1, vocab_size), target.reshape(-1))
            
            # Total loss
            total_model_loss = nll_loss + load_loss
            
            # Accumulate losses
            total_loss += total_model_loss.item()
            total_load_loss += load_loss.item()
            total_nll_loss += nll_loss.item()
            
            # Calculate accuracy
            mask = (target != pad_token_id)
            total_tokens += mask.sum().item()
            predictions = logits.argmax(dim=-1)
            correct_predictions += ((predictions == target) & mask).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    avg_load_loss = total_load_loss / len(dataloader)
    avg_nll_loss = total_nll_loss / len(dataloader)
    accuracy = correct_predictions / total_tokens if total_tokens > 0 else 0
    
    return avg_loss, avg_load_loss, avg_nll_loss, accuracy

### Generation and Evaluation

### Evaluation Metrics and Model Loading

In [ ]:
# Install required packages for evaluation
!pip install rouge-score bert-score nltk transformers

In [ ]:
import re
import string
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import nltk
from datetime import datetime
import uuid

# Download required NLTK data
try:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
except:
    print("NLTK downloads may have failed, but evaluation should still work")

def load_model_from_checkpoint_repo(model_config_key, device='cuda'):
    """
    Load a model from the checkpoint repository mapping.
    
    Args:
        model_config_key: Key from CHECKPOINT_REPO_MAPPING (e.g., 'token_choice_lora')
        device: Device to load model on
        
    Returns:
        model: Loaded model
        model_info: Dictionary with model configuration info
    """
    if model_config_key not in Config.CHECKPOINT_REPO_MAPPING:
        raise ValueError(f"Model config '{model_config_key}' not found in CHECKPOINT_REPO_MAPPING")
    
    repo_id = Config.CHECKPOINT_REPO_MAPPING[model_config_key]
    
    # Parse model configuration from key
    use_gqa = 'gqa' in model_config_key
    use_lora = 'lora' in model_config_key
    use_load_balancer = 'with_LB' in model_config_key
    router_type = 'hash' if 'hash' in model_config_key else 'token_choice'
    
    model_info = {
        'repo_id': repo_id,
        'config_key': model_config_key,
        'use_gqa': use_gqa,
        'use_lora': use_lora,
        'use_load_balancer': use_load_balancer,
        'router_type': router_type
    }
    
    print(f"Loading model: {model_config_key}")
    print(f"  Repository: {repo_id}")
    print(f"  Configuration: GQA={use_gqa}, LoRA={use_lora}, LoadBalancer={use_load_balancer}, Router={router_type}")
    
    # Create model with appropriate configuration
    model = MoETransformer(
        vocab_size=len(tokenizer),
        d_model=Config.D_MODEL,
        nhead=Config.NHEAD,
        num_enc=Config.NUM_ENCODER_LAYERS,
        num_dec=Config.NUM_DECODER_LAYERS,
        num_experts=Config.NUM_EXPERTS,
        d_ff=Config.D_FF,
        router_type=router_type,
        top_k=Config.TOP_K,
        dropout=Config.DROPOUT,
        max_len=Config.MAX_SEQ_LEN,
        use_gqa=use_gqa,
        use_lora=use_lora
    )
    
    # Try to load from HuggingFace
    try:
        checkpoint_filename = f"{model_config_key.replace('_', '-').title()}_best.pt"
        model, epoch, val_loss = load_checkpoint_from_hf(model, repo_id, checkpoint_filename)
        model_info['epoch'] = epoch
        model_info['val_loss'] = val_loss
        model_info['loaded_successfully'] = True
    except Exception as e:
        print(f"Warning: Could not load checkpoint: {e}")
        print("Using randomly initialized model")
        model_info['epoch'] = 0
        model_info['val_loss'] = float('inf')
        model_info['loaded_successfully'] = False
    
    model.to(device)
    model.eval()
    
    return model, model_info

def calculate_rouge_scores(predictions, references):
    """Calculate ROUGE-1, ROUGE-2, and ROUGE-L scores."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        for metric in rouge_scores:
            rouge_scores[metric].append(scores[metric].fmeasure)
    
    # Calculate averages
    avg_scores = {}
    for metric in rouge_scores:
        avg_scores[metric] = {
            'mean': sum(rouge_scores[metric]) / len(rouge_scores[metric]),
            'scores': rouge_scores[metric]
        }
    
    return avg_scores

def calculate_bleu_scores(predictions, references):
    """Calculate BLEU scores using smoothing function."""
    smoothing = SmoothingFunction().method1
    bleu_scores = []
    
    for pred, ref in zip(predictions, references):
        # Tokenize
        pred_tokens = pred.split()
        ref_tokens = [ref.split()]  # BLEU expects list of reference tokenizations
        
        # Calculate BLEU score
        try:
            bleu = sentence_bleu(ref_tokens, pred_tokens, smoothing_function=smoothing)
            bleu_scores.append(bleu)
        except:
            bleu_scores.append(0.0)
    
    return {
        'mean': sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0,
        'scores': bleu_scores
    }

def calculate_bert_scores(predictions, references):
    """Calculate BERTScore using bert-score library."""
    try:
        P, R, F1 = bert_score(predictions, references, lang='en', verbose=False)
        return {
            'precision': {'mean': P.mean().item(), 'scores': P.tolist()},
            'recall': {'mean': R.mean().item(), 'scores': R.tolist()},
            'f1': {'mean': F1.mean().item(), 'scores': F1.tolist()}
        }
    except Exception as e:
        print(f"BERTScore calculation failed: {e}")
        return {
            'precision': {'mean': 0.0, 'scores': [0.0] * len(predictions)},
            'recall': {'mean': 0.0, 'scores': [0.0] * len(predictions)},
            'f1': {'mean': 0.0, 'scores': [0.0] * len(predictions)}
        }

def calculate_compression_ratio(summaries, documents):
    """Calculate compression ratio (summary length / document length)."""
    ratios = []
    for summary, document in zip(summaries, documents):
        doc_len = len(document.split())
        sum_len = len(summary.split())
        ratio = sum_len / doc_len if doc_len > 0 else 0.0
        ratios.append(ratio)
    
    return {
        'mean': sum(ratios) / len(ratios) if ratios else 0.0,
        'scores': ratios
    }

def get_functional_words():
    """Get list of common functional words to exclude from extractiveness calculation."""
    try:
        from nltk.corpus import stopwords
        functional_words = set(stopwords.words('english'))
    except:
        # Fallback list of common functional words
        functional_words = {
            'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from',
            'has', 'he', 'in', 'is', 'it', 'its', 'of', 'on', 'that', 'the',
            'to', 'was', 'were', 'will', 'with', 'would', 'this', 'these', 'those',
            'i', 'you', 'we', 'they', 'me', 'him', 'her', 'them', 'my', 'your',
            'his', 'their', 'our'
        }
    
    # Add punctuation
    functional_words.update(set(string.punctuation))
    return functional_words

def calculate_extractiveness(summaries, documents):
    """Calculate extractiveness: percentage of content words in summary that overlap with document."""
    functional_words = get_functional_words()
    extractiveness_scores = []
    
    for summary, document in zip(summaries, documents):
        # Normalize and tokenize
        summary_words = re.findall(r'\b\w+\b', summary.lower())
        document_words = re.findall(r'\b\w+\b', document.lower())
        
        # Remove functional words
        summary_content = [w for w in summary_words if w not in functional_words]
        document_content = set(w for w in document_words if w not in functional_words)
        
        # Calculate overlap
        if len(summary_content) == 0:
            extractiveness = 0.0
        else:
            overlapping_words = sum(1 for word in summary_content if word in document_content)
            extractiveness = overlapping_words / len(summary_content)
        
        extractiveness_scores.append(extractiveness)
    
    return {
        'mean': sum(extractiveness_scores) / len(extractiveness_scores) if extractiveness_scores else 0.0,
        'scores': extractiveness_scores
    }

def generate_summaries(model, tokenizer, documents, max_length=100, device='cuda'):
    """Generate summaries for a list of documents using the model."""
    model.eval()
    summaries = []
    
    with torch.no_grad():
        for doc in tqdm(documents, desc="Generating summaries"):
            # Tokenize input
            inputs = tokenizer(doc, return_tensors='pt', truncation=True, 
                             max_length=Config.MAX_LENGTH, padding=True)
            input_ids = inputs['input_ids'].to(device)
            
            # Generate summary (simplified greedy decoding)
            generated = []
            decoder_input = torch.tensor([[tokenizer.pad_token_id]], device=device)
            
            for _ in range(max_length):
                with torch.no_grad():
                    logits, _ = model(input_ids, decoder_input, use_load_balancer=False)
                    next_token = logits[:, -1, :].argmax(dim=-1)
                    
                    if next_token.item() == tokenizer.eos_token_id:
                        break
                        
                    decoder_input = torch.cat([decoder_input, next_token.unsqueeze(1)], dim=1)
                    generated.append(next_token.item())
            
            # Decode summary
            summary = tokenizer.decode(generated, skip_special_tokens=True)
            summaries.append(summary)
    
    return summaries

def comprehensive_evaluation(model, model_info, test_data, tokenizer, device='cuda', num_samples=100):
    """Run comprehensive evaluation on a model."""
    print(f"\n{'='*60}")
    print(f"COMPREHENSIVE EVALUATION: {model_info['config_key']}")
    print(f"{'='*60}")
    
    # Sample test data
    test_samples = test_data.select(range(min(num_samples, len(test_data))))
    documents = [item['document'] for item in test_samples]
    reference_summaries = [item['summary'] for item in test_samples]
    
    print(f"Evaluating on {len(documents)} samples...")
    
    # Generate summaries
    print("Generating summaries...")
    predicted_summaries = generate_summaries(model, tokenizer, documents, 
                                           max_length=Config.GENERATION_MAX_LENGTH, device=device)
    
    # Calculate metrics
    print("Calculating metrics...")
    
    # 1. ROUGE Scores
    rouge_scores = calculate_rouge_scores(predicted_summaries, reference_summaries)
    
    # 2. BLEU Score  
    bleu_scores = calculate_bleu_scores(predicted_summaries, reference_summaries)
    
    # 3. BERTScore
    bert_scores = calculate_bert_scores(predicted_summaries, reference_summaries)
    
    # 4. Compression Ratio
    compression_scores = calculate_compression_ratio(predicted_summaries, documents)
    
    # 5. Extractiveness
    extractiveness_scores = calculate_extractiveness(predicted_summaries, documents)
    
    # Compile results
    evaluation_results = {
        'model_info': model_info,
        'metrics': {
            'rouge': rouge_scores,
            'bleu': bleu_scores,
            'bert_score': bert_scores,
            'compression_ratio': compression_scores,
            'extractiveness': extractiveness_scores
        },
        'sample_outputs': {
            'documents': documents[:3],
            'reference_summaries': reference_summaries[:3],
            'predicted_summaries': predicted_summaries[:3]
        },
        'evaluation_timestamp': datetime.now().isoformat(),
        'num_samples': len(documents)
    }
    
    return evaluation_results

def print_evaluation_summary(results):
    """Print a formatted summary of evaluation results."""
    model_info = results['model_info']
    metrics = results['metrics']
    
    print(f"\n📊 EVALUATION SUMMARY: {model_info['config_key'].upper()}")
    print(f"Repository: {model_info['repo_id']}")
    print(f"Configuration: GQA={model_info['use_gqa']}, LoRA={model_info['use_lora']}, LoadBalancer={model_info['use_load_balancer']}")
    print(f"Samples evaluated: {results['num_samples']}")
    print("-" * 50)
    
    # Lexical metrics
    print("📝 LEXICAL METRICS:")
    print(f"  ROUGE-1: {metrics['rouge']['rouge1']['mean']:.4f}")
    print(f"  ROUGE-2: {metrics['rouge']['rouge2']['mean']:.4f}")
    print(f"  ROUGE-L: {metrics['rouge']['rougeL']['mean']:.4f}")
    print(f"  BLEU:    {metrics['bleu']['mean']:.4f}")
    
    # Embedding metrics
    print("\n🤖 EMBEDDING METRICS:")
    print(f"  BERTScore F1: {metrics['bert_score']['f1']['mean']:.4f}")
    print(f"  BERTScore P:  {metrics['bert_score']['precision']['mean']:.4f}")
    print(f"  BERTScore R:  {metrics['bert_score']['recall']['mean']:.4f}")
    
    # Document metrics
    print("\n📄 DOCUMENT METRICS:")
    print(f"  Compression Ratio: {metrics['compression_ratio']['mean']:.4f}")
    print(f"  Extractiveness:    {metrics['extractiveness']['mean']:.4f}")

def save_evaluation_outputs(results, output_dir="evaluation_outputs"):
    """Save evaluation results and sample outputs to files."""
    import os
    import json
    
    os.makedirs(output_dir, exist_ok=True)
    
    model_key = results['model_info']['config_key']
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save full results as JSON
    results_file = os.path.join(output_dir, f"{model_key}_evaluation_{timestamp}.json")
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    # Save sample outputs as text
    samples_file = os.path.join(output_dir, f"{model_key}_samples_{timestamp}.txt")
    with open(samples_file, 'w') as f:
        f.write(f"SAMPLE OUTPUTS - {model_key.upper()}\n")
        f.write("="*60 + "\n\n")
        
        for i in range(3):
            f.write(f"SAMPLE {i+1}:\n")
            f.write("-" * 40 + "\n")
            f.write(f"DOCUMENT:\n{results['sample_outputs']['documents'][i]}\n\n")
            f.write(f"REFERENCE SUMMARY:\n{results['sample_outputs']['reference_summaries'][i]}\n\n")
            f.write(f"PREDICTED SUMMARY:\n{results['sample_outputs']['predicted_summaries'][i]}\n\n")
            f.write("="*60 + "\n\n")
    
    print(f"✅ Results saved to: {results_file}")
    print(f"✅ Sample outputs saved to: {samples_file}")
    
    return results_file, samples_file

In [ ]:
def run_evaluation_suite(test_data, tokenizer, device='cuda', num_samples=100, models_to_evaluate=None):
    """
    Run evaluation on multiple models from the checkpoint repository mapping.
    
    Args:
        test_data: Test dataset
        tokenizer: Tokenizer to use
        device: Device for model inference
        num_samples: Number of test samples to evaluate
        models_to_evaluate: List of model keys to evaluate. If None, evaluates all.
    
    Returns:
        Dict mapping model keys to evaluation results
    """
    if models_to_evaluate is None:
        models_to_evaluate = list(Config.CHECKPOINT_REPO_MAPPING.keys())
    
    print(f"🚀 STARTING EVALUATION SUITE")
    print(f"Models to evaluate: {len(models_to_evaluate)}")
    print(f"Samples per model: {num_samples}")
    print(f"Device: {device}")
    print("="*60)
    
    all_results = {}
    
    for i, model_key in enumerate(models_to_evaluate, 1):
        print(f"\n[{i}/{len(models_to_evaluate)}] Evaluating model: {model_key}")
        
        try:
            # Load model
            model, model_info = load_model_from_checkpoint_repo(model_key, device)
            
            # Run evaluation
            results = comprehensive_evaluation(model, model_info, test_data, tokenizer, device, num_samples)
            
            # Print summary
            print_evaluation_summary(results)
            
            # Save outputs
            save_evaluation_outputs(results)
            
            all_results[model_key] = results
            
            # Clean up model to free GPU memory
            del model
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            
        except Exception as e:
            print(f"❌ Failed to evaluate {model_key}: {str(e)}")
            all_results[model_key] = {'error': str(e), 'model_key': model_key}
        
        print(f"✅ Completed evaluation for {model_key}")
    
    # Generate comparison report
    generate_comparison_report(all_results)
    
    return all_results

def generate_comparison_report(all_results):
    """Generate a comprehensive comparison report across all models."""
    print("\n" + "="*80)
    print("📈 COMPREHENSIVE MODEL COMPARISON REPORT")
    print("="*80)
    
    successful_results = {k: v for k, v in all_results.items() if 'error' not in v}
    
    if not successful_results:
        print("❌ No successful evaluations to compare.")
        return
    
    # Create comparison table
    print(f"\n{'Model':<25} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10} {'BLEU':<10} {'BERTScore':<12} {'Compression':<12} {'Extractive':<12}")
    print("-" * 120)
    
    for model_key, results in successful_results.items():
        if 'metrics' in results:
            metrics = results['metrics']
            model_name = model_key.replace('_', '-').upper()[:24]
            rouge1 = metrics['rouge']['rouge1']['mean']
            rouge2 = metrics['rouge']['rouge2']['mean']
            rougeL = metrics['rouge']['rougeL']['mean']
            bleu = metrics['bleu']['mean']
            bert_f1 = metrics['bert_score']['f1']['mean']
            compression = metrics['compression_ratio']['mean']
            extractive = metrics['extractiveness']['mean']
            
            print(f"{model_name:<25} {rouge1:<10.4f} {rouge2:<10.4f} {rougeL:<10.4f} {bleu:<10.4f} {bert_f1:<12.4f} {compression:<12.4f} {extractive:<12.4f}")
    
    # Find best performing models
    print("\n🏆 BEST PERFORMING MODELS:")
    
    metrics_to_check = {
        'ROUGE-1': lambda r: r['metrics']['rouge']['rouge1']['mean'],
        'ROUGE-2': lambda r: r['metrics']['rouge']['rouge2']['mean'],
        'ROUGE-L': lambda r: r['metrics']['rouge']['rougeL']['mean'],
        'BLEU': lambda r: r['metrics']['bleu']['mean'],
        'BERTScore F1': lambda r: r['metrics']['bert_score']['f1']['mean']
    }
    
    for metric_name, metric_func in metrics_to_check.items():
        try:
            best_model = max(successful_results.items(), 
                           key=lambda x: metric_func(x[1]) if 'metrics' in x[1] else 0)
            best_score = metric_func(best_model[1])
            print(f"  {metric_name}: {best_model[0]} ({best_score:.4f})")
        except:
            print(f"  {metric_name}: Unable to determine best model")
    
    # Configuration analysis
    print(f"\n🔍 CONFIGURATION ANALYSIS:")
    
    # Group by configuration features
    gqa_models = [k for k, v in successful_results.items() if v.get('model_info', {}).get('use_gqa', False)]
    lora_models = [k for k, v in successful_results.items() if v.get('model_info', {}).get('use_lora', False)]
    lb_models = [k for k, v in successful_results.items() if v.get('model_info', {}).get('use_load_balancer', False)]
    
    print(f"  Models with GQA: {len(gqa_models)}")
    print(f"  Models with LoRA: {len(lora_models)}")
    print(f"  Models with Load Balancer: {len(lb_models)}")
    
    # Save comparison report
    save_comparison_report(all_results)

def save_comparison_report(all_results, output_dir="evaluation_outputs"):
    """Save detailed comparison report to file."""
    import os
    import json
    
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_file = os.path.join(output_dir, f"model_comparison_report_{timestamp}.json")
    
    # Create summary report
    report = {
        'evaluation_timestamp': timestamp,
        'total_models_attempted': len(all_results),
        'successful_evaluations': len([r for r in all_results.values() if 'error' not in r]),
        'failed_evaluations': len([r for r in all_results.values() if 'error' in r]),
        'model_results': all_results,
        'summary_statistics': {}
    }
    
    # Calculate summary statistics
    successful_results = {k: v for k, v in all_results.items() if 'error' not in v}
    
    if successful_results:
        metrics_summary = {}
        metric_paths = [
            ('rouge1', ['metrics', 'rouge', 'rouge1', 'mean']),
            ('rouge2', ['metrics', 'rouge', 'rouge2', 'mean']),
            ('rougeL', ['metrics', 'rouge', 'rougeL', 'mean']),
            ('bleu', ['metrics', 'bleu', 'mean']),
            ('bert_f1', ['metrics', 'bert_score', 'f1', 'mean']),
            ('compression', ['metrics', 'compression_ratio', 'mean']),
            ('extractiveness', ['metrics', 'extractiveness', 'mean'])
        ]
        
        for metric_name, path in metric_paths:
            values = []
            for result in successful_results.values():
                try:
                    value = result
                    for key in path:
                        value = value[key]
                    values.append(value)
                except:
                    continue
            
            if values:
                metrics_summary[metric_name] = {
                    'mean': sum(values) / len(values),
                    'min': min(values),
                    'max': max(values),
                    'count': len(values)
                }
        
        report['summary_statistics'] = metrics_summary
    
    # Save report
    with open(report_file, 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"📊 Comparison report saved to: {report_file}")
    return report_file

# Example usage function
def run_quick_evaluation_demo():
    """
    Demonstration function showing how to use the evaluation system.
    This function is ready to run once you have the test data loaded.
    """
    print("🔬 EVALUATION SYSTEM DEMONSTRATION")
    print("This function shows how to use the evaluation system.")
    print("Uncomment and modify the code below to run actual evaluation.")
    
    # Uncomment and modify these lines to run evaluation:
    """
    # Load test data and tokenizer (assuming they are already loaded)
    # tokenizer = AutoTokenizer.from_pretrained(Config.TOKENIZER_NAME)
    # _, _, test_loader, test_data = load_data(tokenizer)
    
    # Evaluate a single model
    # model_key = 'token_choice_lora'  # Choose from CHECKPOINT_REPO_MAPPING keys
    # model, model_info = load_model_from_checkpoint_repo(model_key)
    # results = comprehensive_evaluation(model, model_info, test_data, tokenizer, device, num_samples=50)
    # print_evaluation_summary(results)
    # save_evaluation_outputs(results)
    
    # Or evaluate all models
    # all_results = run_evaluation_suite(test_data, tokenizer, device='cuda', num_samples=100)
    
    # Evaluate specific subset of models
    # models_to_test = ['token_choice', 'token_choice_lora', 'hash', 'hash_lora']
    # subset_results = run_evaluation_suite(test_data, tokenizer, device='cuda', 
    #                                     num_samples=100, models_to_evaluate=models_to_test)
    """
    
    print("\nAvailable models for evaluation:")
    for i, model_key in enumerate(Config.CHECKPOINT_REPO_MAPPING.keys(), 1):
        repo = Config.CHECKPOINT_REPO_MAPPING[model_key]
        print(f"  {i:2d}. {model_key:<30} -> {repo}")
    
    print(f"\nTotal models available: {len(Config.CHECKPOINT_REPO_MAPPING)}")
    print("\nReady for evaluation! 🚀")

# Display available evaluation functions
print("📋 EVALUATION SYSTEM LOADED")
print("Available functions:")
print("  • load_model_from_checkpoint_repo() - Load specific model")
print("  • comprehensive_evaluation() - Evaluate single model")
print("  • run_evaluation_suite() - Evaluate multiple models")  
print("  • run_quick_evaluation_demo() - Show usage examples")
print(f"  • {len(Config.CHECKPOINT_REPO_MAPPING)} models available for evaluation")

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, metrics, save_path, model_type='base'):
    """Enhanced checkpoint saving with LoRA support."""
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch,
        'metrics': metrics,
        'model_type': model_type,
        'use_lora': getattr(model, 'use_lora', False),
    }
    
    # Add parameter summary for tracking efficiency
    if hasattr(model, 'get_parameter_summary'):
        checkpoint['parameter_summary'] = model.get_parameter_summary()
    
    torch.save(checkpoint, save_path)
    print(f"Checkpoint saved: {save_path}")
    if 'parameter_summary' in checkpoint:
        summary = checkpoint['parameter_summary']
        print(f"Model efficiency: {summary['overall_efficiency']:.2%}")
        if summary.get('use_lora'):
            print(f"MoE LoRA efficiency: {summary['moe_efficiency']:.2%}")

def load_checkpoint(model, optimizer, scheduler, checkpoint_path):
    """Enhanced checkpoint loading with LoRA compatibility."""
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    epoch = checkpoint['epoch']
    metrics = checkpoint['metrics']
    model_type = checkpoint.get('model_type', 'base')
    use_lora = checkpoint.get('use_lora', False)
    
    print(f"Loaded checkpoint from epoch {epoch}, model type: {model_type}")
    if use_lora:
        print("LoRA configuration detected in checkpoint")
    
    if 'parameter_summary' in checkpoint:
        summary = checkpoint['parameter_summary']
        print(f"Model efficiency: {summary['overall_efficiency']:.2%}")
        if summary.get('use_lora'):
            print(f"MoE LoRA efficiency: {summary['moe_efficiency']:.2%}")
    
    return epoch, metrics, model_type

def get_checkpoint_filename(config, run_id, model_type='base', epoch=0, is_best=False):
    """Get standardized checkpoint filename with LoRA support."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if model_type != 'base':
        base_name = f"checkpoint_{model_type}_epoch_{epoch:03d}"
    else:
        base_name = f"checkpoint_epoch_{epoch:03d}"
    
    if is_best:
        base_name += "_best"
    
    base_name += f"_{timestamp}.pt"
    
    checkpoint_dir = os.path.join("checkpoints", model_type, run_id)
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    return os.path.join(checkpoint_dir, base_name)

### Checkpointing

In [ ]:
def save_checkpoint(model, name, epoch, val_loss, is_best=False):
    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'val_loss': val_loss,
        'config': {
            'D_MODEL': Config.D_MODEL,
            'NUM_EXPERTS': Config.NUM_EXPERTS,
            'name': name
        }
    }
    
    filename = f"{name}_best.pt" if is_best else f"{name}_epoch{epoch}.pt"
    path = os.path.join(Config.CHECKPOINT_DIR, filename)
    torch.save(checkpoint, path)
    print(f"Saved checkpoint: {filename}")

def push_to_hub(model, name):
    if not Config.PUSH_TO_HUB:
        return
    
    repo_name = f"{Config.HF_USERNAME}/sparse-moe-{name}"
    try:
        create_repo(repo_name, exist_ok=True)
        api = HfApi()
        
        # Save model
        model_path = f"{Config.CHECKPOINT_DIR}/{name}_best.pt"
        if os.path.exists(model_path):
            api.upload_file(
                path_or_fileobj=model_path,
                path_in_repo=f"{name}_best.pt",
                repo_id=repo_name
            )
            print(f"Pushed to Hub: {repo_name}")
    except Exception as e:
        print(f"Failed to push to Hub: {e}")

In [ ]:
# Initialize model with LoRA support
def initialize_model(config, use_lora=False):
    """Initialize model with optional LoRA support."""
    model = MoETransformer(
        vocab_size=config.VOCAB_SIZE,
        d_model=config.D_MODEL,
        nhead=config.NHEAD,
        num_enc=config.NUM_ENC_LAYERS,
        num_dec=config.NUM_DEC_LAYERS,
        num_experts=config.NUM_EXPERTS,
        d_ff=config.D_FF,
        router_type=config.ROUTER_TYPE,
        top_k=config.TOP_K,
        dropout=config.DROPOUT,
        max_len=config.MAX_LEN,
        use_gqa=config.USE_GQA,
        use_lora=use_lora
    )
    
    # Print parameter summary
    if hasattr(model, 'get_parameter_summary'):
        summary = model.get_parameter_summary()
        print(f"\nModel Parameter Summary:")
        print(f"Total parameters: {summary['total_parameters']:,}")
        print(f"Trainable parameters: {summary['trainable_parameters']:,}")
        print(f"Frozen parameters: {summary['frozen_parameters']:,}")
        print(f"Overall efficiency: {summary['overall_efficiency']:.2%}")
        if summary.get('use_lora'):
            print(f"MoE efficiency with LoRA: {summary['moe_efficiency']:.2%}")
    
    return model

# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Generate run ID
run_id = str(uuid.uuid4())[:8]
print(f"Run ID: {run_id}")

# Model selection
USE_LORA = True  # Set to True to use LoRA, False for standard training
model_type = "lora" if USE_LORA else "base"

print(f"\nInitializing {model_type.upper()} model...")

# Create model
model = initialize_model(Config, use_lora=USE_LORA)
model.to(device)

# Initialize optimizer and scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY,
    betas=(0.9, 0.98)
)

total_steps = len(train_dataloader) * Config.EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Loss function
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

# Training loop with enhanced logging
print(f"\nStarting training for {Config.EPOCHS} epochs...")
print(f"Model type: {model_type}")
print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

best_val_loss = float('inf')
train_losses = []
val_losses = []

for epoch in range(Config.EPOCHS):
    print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
    
    # Training
    train_loss, train_load_loss, train_nll_loss, train_acc = train_epoch(
        model, train_dataloader, optimizer, criterion, device,
        Config.VOCAB_SIZE, tokenizer.pad_token_id, use_load_balancer=True
    )
    
    # Validation
    val_loss, val_load_loss, val_nll_loss, val_acc = evaluate_model(
        model, val_dataloader, criterion, device,
        Config.VOCAB_SIZE, tokenizer.pad_token_id, use_load_balancer=True
    )
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # Current learning rate
    current_lr = scheduler.get_last_lr()[0]
    
    print(f"Train Loss: {train_loss:.4f} (NLL: {train_nll_loss:.4f}, Load: {train_load_loss:.4f})")
    print(f"Val Loss: {val_loss:.4f} (NLL: {val_nll_loss:.4f}, Load: {val_load_loss:.4f})")
    print(f"Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
    print(f"Learning Rate: {current_lr:.6f}")
    
    # Save checkpoint
    metrics = {
        'train_loss': train_loss,
        'train_load_loss': train_load_loss,
        'train_nll_loss': train_nll_loss,
        'train_accuracy': train_acc,
        'val_loss': val_loss,
        'val_load_loss': val_load_loss,
        'val_nll_loss': val_nll_loss,
        'val_accuracy': val_acc,
        'learning_rate': current_lr
    }
    
    # Save regular checkpoint
    checkpoint_path = get_checkpoint_filename(Config, run_id, model_type, epoch, is_best=False)
    save_checkpoint(model, optimizer, scheduler, epoch, metrics, checkpoint_path, model_type)
    
    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_checkpoint_path = get_checkpoint_filename(Config, run_id, model_type, epoch, is_best=True)
        save_checkpoint(model, optimizer, scheduler, epoch, metrics, best_checkpoint_path, model_type)
        print(f"New best model saved! Val loss: {val_loss:.4f}")
    
    scheduler.step()

print(f"\nTraining completed!")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Final model type: {model_type}")
print(f"Run ID: {run_id}")